# Deployment Training Notebook — AE + RF Fusion with 30 Realtime Features

This notebook creates a deployment-ready model package for the realtime SDN-based IDS/IPS prototype.

Main change compared with the old notebook:

- Old pipeline: scaler was fitted on 65 selected features, then the first 30 scaled features were used for the Autoencoder.
- Deployment pipeline: scaler is fitted directly on the 30 realtime-extractable features.
- Autoencoder is retrained on the 30 scaled features.
- Random Forest is retrained on the fusion vector: 20 primary features + 5 latent features = 25 features.

Expected output folder:

```text
models_deploy/
├── scaler.joblib
├── ae_model.pth
├── rf_model.joblib
├── feature_30.json
├── feature_20.json
└── deployment_metadata.json
```

In [1]:
# Cell 1: Setup and Environment Initialization

%load_ext autoreload
%autoreload 2

import sys
import os
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Setup path
current_dir = Path.cwd()
root_dir = current_dir.parent

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

print(f"Project Root: {root_dir}")

# Import project modules
from src import config, preprocessing, autoencoder, rf_classifier, evaluation, utils

print("Checking Data Directories:")
print(f"  - 2017: {config.DIR_2017} -> {'Found' if config.DIR_2017.exists() else 'Not Found'}")
print(f"  - 2018: {config.DIR_2018} -> {'Found' if config.DIR_2018.exists() else 'Not Found'}")

if not config.DIR_2017.exists() or not config.DIR_2018.exists():
    print("WARNING: Please verify the dataset folder paths in src/config.py")

# Output directory for deployment artifacts
deploy_dir = root_dir / "models_deploy"
deploy_dir.mkdir(parents=True, exist_ok=True)

eval_dir = deploy_dir / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)

print(f"Deployment model directory: {deploy_dir}")
print(f"Evaluation output directory: {eval_dir}")
print(f"Device from config: {config.DEVICE}")

Project Root: c:\Users\Admin\Documents\ids_ae_rf_fusion
Checking Data Directories:
  - 2017: C:\Users\Admin\Documents\ids_ae_rf_fusion\datasets\CIC-IDS2017 -> Found
  - 2018: C:\Users\Admin\Documents\ids_ae_rf_fusion\datasets\CSE-CIC-IDS2018 -> Found
Deployment model directory: c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy
Evaluation output directory: c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy\evaluation
Device from config: cuda


In [2]:
# Cell 2: Load 2017 and 2018 Datasets Separately

print("Loading 2017 and 2018 datasets separately...")

X_17, y_17 = preprocessing.load_single_dataset_year("2017", binary_mode=True)
print(f"Loaded 2017: X={X_17.shape}, y={y_17.shape}")

X_18, y_18 = preprocessing.load_single_dataset_year("2018", binary_mode=True)
print(f"Loaded 2018: X={X_18.shape}, y={y_18.shape}")

# Ensure X dataframes follow config.SELECTED_FEATURES order
assert list(X_17.columns) == config.SELECTED_FEATURES, "X_17 columns do not match config.SELECTED_FEATURES"
assert list(X_18.columns) == config.SELECTED_FEATURES, "X_18 columns do not match config.SELECTED_FEATURES"

print("Feature order verified: both datasets follow config.SELECTED_FEATURES.")

Loading 2017 and 2018 datasets separately...
🔄 Loading dataset year 2017 (Binary=True)...
✅ Loaded 2017. Shape: (2830743, 65)
Loaded 2017: X=(2830743, 65), y=(2830743,)
🔄 Loading dataset year 2018 (Binary=True)...
✅ Loaded 2018. Shape: (9625148, 65)
Loaded 2018: X=(9625148, 65), y=(9625148,)
Feature order verified: both datasets follow config.SELECTED_FEATURES.


In [3]:
# Cell 3: Verify Deployment Feature Lists

FEATURE_30 = list(config.AE_INPUT_FEATURES)
FEATURE_20 = list(config.mRMR_FEATURES)

print(f"Number of full selected features in config.SELECTED_FEATURES: {len(config.SELECTED_FEATURES)}")
print(f"Number of deployment AE input features: {len(FEATURE_30)}")
print(f"Number of primary fusion features: {len(FEATURE_20)}")

missing_30 = [f for f in FEATURE_30 if f not in config.SELECTED_FEATURES]
missing_20 = [f for f in FEATURE_20 if f not in FEATURE_30]

if missing_30:
    raise ValueError(f"These FEATURE_30 items are missing from SELECTED_FEATURES: {missing_30}")

if missing_20:
    raise ValueError(f"These FEATURE_20 items are missing from FEATURE_30: {missing_20}")

print("Deployment feature lists are valid.")

print("\nFirst 10 FEATURE_30:")
for f in FEATURE_30[:10]:
    print(" -", f)

print("\nFEATURE_20:")
for f in FEATURE_20:
    print(" -", f)

Number of full selected features in config.SELECTED_FEATURES: 65
Number of deployment AE input features: 30
Number of primary fusion features: 20
Deployment feature lists are valid.

First 10 FEATURE_30:
 - RST Flag Count
 - Total Length of Fwd Packets
 - Bwd IAT Min
 - ECE Flag Count
 - act_data_pkt_fwd
 - Idle Std
 - Bwd Packet Length Min
 - Total Fwd Packets
 - Bwd IAT Mean
 - PSH Flag Count

FEATURE_20:
 - RST Flag Count
 - Total Length of Fwd Packets
 - Bwd IAT Min
 - ECE Flag Count
 - act_data_pkt_fwd
 - Idle Std
 - Bwd Packet Length Min
 - Total Fwd Packets
 - Bwd IAT Mean
 - PSH Flag Count
 - Destination Port
 - Flow IAT Std
 - Bwd Packet Length Std
 - Bwd IAT Max
 - Fwd Packet Length Max
 - Fwd PSH Flags
 - Active Min
 - Init_Win_bytes_backward
 - Bwd Packets/s
 - Fwd IAT Min


In [4]:
# Cell 4: Split Datasets and Construct Mixed Training Set

print("Splitting 2017 and 2018 independently with stratification...")

X_17_train, X_17_test, y_17_train, y_17_test = train_test_split(
    X_17,
    y_17,
    test_size=0.2,
    random_state=config.SEED,
    stratify=y_17
)

X_18_train, X_18_test, y_18_train, y_18_test = train_test_split(
    X_18,
    y_18,
    test_size=0.2,
    random_state=config.SEED,
    stratify=y_18
)

# Keep DataFrames instead of converting to numpy.
# This avoids feature-order confusion during 30-feature deployment training.
X_train_df = pd.concat([X_17_train, X_18_train], ignore_index=True)
y_train = pd.concat([pd.Series(y_17_train), pd.Series(y_18_train)], ignore_index=True)

X_test_all_df = pd.concat([X_17_test, X_18_test], ignore_index=True)
y_test_all = pd.concat([pd.Series(y_17_test), pd.Series(y_18_test)], ignore_index=True)

print(f"Mixed Train Size: {X_train_df.shape}")
print(f"Test 2017 Size:  {X_17_test.shape}")
print(f"Test 2018 Size:  {X_18_test.shape}")
print(f"Global Test Size: {X_test_all_df.shape}")

print("\nTraining label distribution:")
print(y_train.value_counts())

# Free some memory
del X_17, X_18, X_17_train, X_18_train
gc.collect()

Splitting 2017 and 2018 independently with stratification...
Mixed Train Size: (9964712, 65)
Test 2017 Size:  (566149, 65)
Test 2018 Size:  (1925030, 65)
Global Test Size: (2491179, 65)

Training label distribution:
Label
0    7320007
1    2644705
Name: count, dtype: int64


30

In [5]:
# Cell 5: Select 30 Realtime-Deployable Features

print("Selecting 30 deployment features...")

X_train_30 = X_train_df[FEATURE_30].copy()
X_17_test_30 = X_17_test[FEATURE_30].copy()
X_18_test_30 = X_18_test[FEATURE_30].copy()
X_test_all_30 = X_test_all_df[FEATURE_30].copy()

print(f"X_train_30: {X_train_30.shape}")
print(f"X_17_test_30: {X_17_test_30.shape}")
print(f"X_18_test_30: {X_18_test_30.shape}")
print(f"X_test_all_30: {X_test_all_30.shape}")

assert X_train_30.shape[1] == 30
assert list(X_train_30.columns) == FEATURE_30

print("30-feature deployment matrices are ready.")

Selecting 30 deployment features...
X_train_30: (9964712, 30)
X_17_test_30: (566149, 30)
X_18_test_30: (1925030, 30)
X_test_all_30: (2491179, 30)
30-feature deployment matrices are ready.


In [6]:
# Cell 6: Fit and Save 30-Feature Scaler

print("Scaling 30 selected features for deployment...")

scaler_30 = StandardScaler()

X_train_30_scaled = scaler_30.fit_transform(X_train_30)
X_17_test_30_scaled = scaler_30.transform(X_17_test_30)
X_18_test_30_scaled = scaler_30.transform(X_18_test_30)
X_test_all_30_scaled = scaler_30.transform(X_test_all_30)

scaler_path = deploy_dir / "scaler.joblib"
joblib.dump(scaler_30, scaler_path)

print(f"Saved 30-feature scaler to: {scaler_path}")
print("Scaler n_features_in_:", getattr(scaler_30, "n_features_in_", None))

Scaling 30 selected features for deployment...
Saved 30-feature scaler to: c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy\scaler.joblib
Scaler n_features_in_: 30


In [7]:
# Cell 7: Train Autoencoder on 30 Scaled Features

print(f"Training Autoencoder on device: {config.DEVICE}")

L = 30
N = 5

ae_model = autoencoder.DeepAutoencoder(
    input_dim=L,
    latent_dim=N,
    hidden_layers=[22, 12]
)

ae_save_path = deploy_dir / "ae_model.pth"

ae_model = autoencoder.train_ae(
    ae_model,
    X_train_30_scaled,
    save_path=ae_save_path
)

print("AE training completed.")
print(f"Saved AE model to: {ae_save_path}")

Training Autoencoder on device: cuda
[Autoencoder] Training on cuda...
  Epoch 5/30 - Loss: 0.223714
  Epoch 10/30 - Loss: 0.126750
  Epoch 15/30 - Loss: 0.092470
  Epoch 20/30 - Loss: 0.085157
  Epoch 25/30 - Loss: 0.082537
  Epoch 30/30 - Loss: 0.080501
✅ AE Model saved to: c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy\ae_model.pth
AE training completed.
Saved AE model to: c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy\ae_model.pth


In [8]:
# Cell 8: Extract Latent Features from Autoencoder Encoder

print("Extracting 5-dimensional latent features...")

X_train_latent = autoencoder.extract_features(ae_model, X_train_30_scaled)
X_17_test_latent = autoencoder.extract_features(ae_model, X_17_test_30_scaled)
X_18_test_latent = autoencoder.extract_features(ae_model, X_18_test_30_scaled)
X_test_all_latent = autoencoder.extract_features(ae_model, X_test_all_30_scaled)

print(f"X_train_latent: {X_train_latent.shape}")
print(f"X_17_test_latent: {X_17_test_latent.shape}")
print(f"X_18_test_latent: {X_18_test_latent.shape}")
print(f"X_test_all_latent: {X_test_all_latent.shape}")

assert X_train_latent.shape[1] == 5

Extracting 5-dimensional latent features...
X_train_latent: (9964712, 5)
X_17_test_latent: (566149, 5)
X_18_test_latent: (1925030, 5)
X_test_all_latent: (2491179, 5)


In [9]:
# Cell 9: Fusion Strategy — 20 Primary Features + 5 Latent Features

print("Executing fusion strategy: 20 primary features + 5 latent features...")

# FEATURE_20 is a selected subset of FEATURE_30.
idx_20 = [FEATURE_30.index(f) for f in FEATURE_20]

X_train_20_scaled = X_train_30_scaled[:, idx_20]
X_17_test_20_scaled = X_17_test_30_scaled[:, idx_20]
X_18_test_20_scaled = X_18_test_30_scaled[:, idx_20]
X_test_all_20_scaled = X_test_all_30_scaled[:, idx_20]

X_train_fusion = np.hstack([X_train_20_scaled, X_train_latent])
X_17_test_fusion = np.hstack([X_17_test_20_scaled, X_17_test_latent])
X_18_test_fusion = np.hstack([X_18_test_20_scaled, X_18_test_latent])
X_test_all_fusion = np.hstack([X_test_all_20_scaled, X_test_all_latent])

print(f"X_train_fusion: {X_train_fusion.shape}")
print(f"X_17_test_fusion: {X_17_test_fusion.shape}")
print(f"X_18_test_fusion: {X_18_test_fusion.shape}")
print(f"X_test_all_fusion: {X_test_all_fusion.shape}")

assert X_train_fusion.shape[1] == 25

Executing fusion strategy: 20 primary features + 5 latent features...
X_train_fusion: (9964712, 25)
X_17_test_fusion: (566149, 25)
X_18_test_fusion: (1925030, 25)
X_test_all_fusion: (2491179, 25)


In [10]:
# Cell 10: Train Random Forest on 25-Dimensional Fusion Vector

print("Training Random Forest on fusion data...")

rf_save_path = deploy_dir / "rf_model.joblib"

rf_model = rf_classifier.train_rf(
    X_train_fusion,
    y_train,
    save_path=rf_save_path
)

print("RF training completed.")
print(f"Saved RF model to: {rf_save_path}")
print("RF n_features_in_:", getattr(rf_model, "n_features_in_", None))

Training Random Forest on fusion data...
[RandomForest] Training classifier...


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


building tree 1 of 100building tree 2 of 100
building tree 3 of 100

building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100


[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  2.4min


building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75

[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:  9.0min finished


In [11]:
# Cell 11: Evaluate Deployment Model

print("\n--- STARTING DEPLOYMENT MODEL EVALUATION ---")

test_scenarios = [
    (X_17_test_fusion, y_17_test, "Deploy30_Unseen_2017"),
    (X_18_test_fusion, y_18_test, "Deploy30_Unseen_2018"),
    (X_test_all_fusion, y_test_all, "Deploy30_Global_Mixed_Test"),
]

for X_t, y_t, name in test_scenarios:
    print(f"\nEvaluating Scenario: {name}")
    evaluation.evaluate_model(
        model=rf_model,
        X_test=X_t,
        y_test=y_t,
        save_dir=eval_dir,
        dataset_name=name
    )


--- STARTING DEPLOYMENT MODEL EVALUATION ---

Evaluating Scenario: Deploy30_Unseen_2017

Evaluating on Deploy30_Unseen_2017...


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.4s finished


Accuracy: 0.9907
 MCC:      0.9715
Report saved to: report_Deploy30_Unseen_2017.txt
Confusion Matrix saved to: cm_Deploy30_Unseen_2017.png

Evaluating Scenario: Deploy30_Unseen_2018

Evaluating on Deploy30_Unseen_2018...


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.6s finished


Accuracy: 0.9585
 MCC:      0.9015
Report saved to: report_Deploy30_Unseen_2018.txt
Confusion Matrix saved to: cm_Deploy30_Unseen_2018.png

Evaluating Scenario: Deploy30_Global_Mixed_Test

Evaluating on Deploy30_Global_Mixed_Test...


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.1s finished


Accuracy: 0.9658
 MCC:      0.9151
Report saved to: report_Deploy30_Global_Mixed_Test.txt
Confusion Matrix saved to: cm_Deploy30_Global_Mixed_Test.png


In [12]:
# Cell 12: Save Feature Lists and Deployment Metadata

feature_30_path = deploy_dir / "feature_30.json"
feature_20_path = deploy_dir / "feature_20.json"
metadata_path = deploy_dir / "deployment_metadata.json"

with open(feature_30_path, "w", encoding="utf-8") as f:
    json.dump(FEATURE_30, f, indent=2, ensure_ascii=False)

with open(feature_20_path, "w", encoding="utf-8") as f:
    json.dump(FEATURE_20, f, indent=2, ensure_ascii=False)

metadata = {
    "description": "Deployment-ready AE-RF fusion model using 30 realtime-extractable features.",
    "scaler_input_dim": 30,
    "ae_input_dim": 30,
    "ae_latent_dim": 5,
    "rf_input_dim": 25,
    "fusion": "20 primary scaled features + 5 Autoencoder latent features",
    "feature_30": FEATURE_30,
    "feature_20": FEATURE_20,
    "seed": config.SEED,
    "rf_estimators": config.RF_ESTIMATORS,
    "rf_max_depth": config.RF_MAX_DEPTH,
    "ae_hidden_layers": [22, 12],
    "ae_epochs": config.AE_EPOCHS,
    "ae_batch_size": config.AE_BATCH_SIZE,
    "ae_lr": config.AE_LR,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved:")
print(" -", feature_30_path)
print(" -", feature_20_path)
print(" -", metadata_path)

Saved:
 - c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy\feature_30.json
 - c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy\feature_20.json
 - c:\Users\Admin\Documents\ids_ae_rf_fusion\models_deploy\deployment_metadata.json


In [13]:
# Cell 13: Sanity Check Saved Artifacts

print("Running sanity check on saved deployment artifacts...")

loaded_scaler = joblib.load(deploy_dir / "scaler.joblib")
loaded_rf = joblib.load(deploy_dir / "rf_model.joblib")

print("Scaler n_features_in_:", getattr(loaded_scaler, "n_features_in_", None))
print("RF n_features_in_:", getattr(loaded_rf, "n_features_in_", None))

assert getattr(loaded_scaler, "n_features_in_", None) == 30, "Scaler must expect 30 features."
assert getattr(loaded_rf, "n_features_in_", None) == 25, "RF must expect 25 fused features."

state = torch.load(deploy_dir / "ae_model.pth", map_location="cpu")
print("\nAE state_dict shapes:")
for k, v in state.items():
    if hasattr(v, "shape"):
        print(k, tuple(v.shape))

print("\nDeployment artifacts are consistent.")
print(f"Copy this folder to the VM if needed: {deploy_dir}")

Running sanity check on saved deployment artifacts...
Scaler n_features_in_: 30
RF n_features_in_: 25

AE state_dict shapes:
encoder.0.weight (22, 30)
encoder.0.bias (22,)
encoder.1.weight (22,)
encoder.1.bias (22,)
encoder.1.running_mean (22,)
encoder.1.running_var (22,)
encoder.1.num_batches_tracked ()
encoder.3.weight (12, 22)
encoder.3.bias (12,)
encoder.4.weight (12,)
encoder.4.bias (12,)
encoder.4.running_mean (12,)
encoder.4.running_var (12,)
encoder.4.num_batches_tracked ()
encoder.6.weight (5, 12)
encoder.6.bias (5,)
decoder.0.weight (12, 5)
decoder.0.bias (12,)
decoder.1.weight (12,)
decoder.1.bias (12,)
decoder.1.running_mean (12,)
decoder.1.running_var (12,)
decoder.1.num_batches_tracked ()
decoder.3.weight (22, 12)
decoder.3.bias (22,)
decoder.4.weight (22,)
decoder.4.bias (22,)
decoder.4.running_mean (22,)
decoder.4.running_var (22,)
decoder.4.num_batches_tracked ()
decoder.6.weight (30, 22)
decoder.6.bias (30,)

Deployment artifacts are consistent.
Copy this folder to th

In [14]:
# Cell 14: Realtime Predictor Command

print("After copying models_deploy/ to the VM, run this in the realtime folder:")
print()
print("python3 predictor.py \\")
print("  --scaler ../models_deploy/scaler.joblib \\")
print("  --ae ../models_deploy/ae_model.pth \\")
print("  --rf ../models_deploy/rf_model.joblib \\")
print("  --dry-run")

After copying models_deploy/ to the VM, run this in the realtime folder:

python3 predictor.py \
  --scaler ../models_deploy/scaler.joblib \
  --ae ../models_deploy/ae_model.pth \
  --rf ../models_deploy/rf_model.joblib \
  --dry-run
